# Go Sales Data Pipeline - Silver Layer Transformation

This notebook transforms bronze layer data into clean, business-ready silver tables.

**Data Source:** Bronze Delta tables from ADLS ingestion (`datalakedegroup3/gosales/raw/`)

**Objectives:**
* Standardize formats (trim, initcap, upper)
* Handle null values using business rules (imputation by brand/type)
* Remove duplicates on primary keys
* Correct invalid values (unit_cost > unit_price)
* Add derived columns (margin_pct, revenue, discount)
* Implement SCD Type 2 for product dimension history
* Optimize Delta tables with ZORDER

**Output Tables:**
* `silver_products` — Cleaned products with SCD Type 2 history (270 records)
* `silver_retailers` — Standardized retailer dimension
* `silver_order_methods` — Cleaned order method types
* `silver_daily_sales` — Enriched sales with revenue & date dimensions

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.window import Window

schema = "default"

# Silver layer reads from Delta tables (Hive metastore, no Unity Catalog)
# ADLS access is only needed in the Bronze notebook for raw file ingestion
print(f"Schema: {schema}")

Schema: default
Silver path: abfss://gosales@datalakedegroup3.dfs.core.windows.net/silver
Gold path: abfss://gosales@datalakedegroup3.dfs.core.windows.net/gold


## 1. Products Silver Transformation

**Source:** `bronze_products` (282 records)  
**Target:** `silver_products`

Apply standardization, null imputation, cost corrections, and deduplication.

In [0]:
bronze_products = spark.table(
    f"{schema}.bronze_products"
)

products_df = (
    bronze_products

    .withColumn(
        "product_number",
        upper(trim(col("Product_number")))
    )

    .withColumn(
        "product_line",
        initcap(trim(col("Product_line")))
    )

    .withColumn(
        "product_type",
        initcap(trim(col("Product_type")))
    )

    .withColumn(
        "product_name",
        initcap(trim(col("Product")))
    )

    .withColumn(
        "product_brand",
        initcap(trim(col("Product_brand")))
    )

    .withColumn(
        "product_color",
        when(lower(trim(col("Product_color")))=="blu","Blue")
        .when(lower(trim(col("Product_color")))=="redd","Red")
        .otherwise(initcap(trim(col("Product_color"))))
    )

    .withColumnRenamed("Unit_cost","unit_cost")
    .withColumnRenamed("Unit_price","unit_price")
)

### 1.1 Product Type Imputation

Fill null `product_type` values using the most frequent type for the same `product_brand`.

In [0]:
type_lookup = (
    products_df
    .filter(col("product_type").isNotNull())
    .groupBy("product_brand","product_type")
    .count()
)

w = Window.partitionBy("product_brand")\
.orderBy(desc("count"))

type_lookup = (
    type_lookup
    .withColumn("rn",row_number().over(w))
    .filter(col("rn")==1)
    .select(
        "product_brand",
        col("product_type").alias("fill_type")
    )
)

products_df = (
    products_df
    .join(type_lookup,"product_brand","left")
    .withColumn(
        "product_type",
        coalesce(
            col("product_type"),
            col("fill_type")
        )
    )
    .drop("fill_type")
)

### 1.2 Product Line Imputation

Fill null `product_line` values using the most frequent line for the same `product_brand`.

In [0]:
line_lookup = (
    products_df
    .filter(col("product_line").isNotNull())
    .groupBy("product_brand","product_line")
    .count()
)

w = Window.partitionBy("product_brand")\
.orderBy(desc("count"))

line_lookup = (
    line_lookup
    .withColumn("rn",row_number().over(w))
    .filter(col("rn")==1)
    .select(
        "product_brand",
        col("product_line").alias("fill_line")
    )
)

products_df = (
    products_df
    .join(line_lookup,"product_brand","left")
    .withColumn(
        "product_line",
        coalesce(
            col("product_line"),
            col("fill_line")
        )
    )
    .drop("fill_line")
)

### 1.3 Product Name Imputation

Fill null `product_name` values using the first non-null name within the same `product_brand`.

In [0]:
name_lookup = (
    products_df
    .filter(col("product_name").isNotNull())
    .groupBy("product_brand")
    .agg(
        first("product_name").alias("fill_name")
    )
)

products_df = (
    products_df
    .join(name_lookup,"product_brand","left")
    .withColumn(
        "product_name",
        coalesce(
            col("product_name"),
            col("fill_name")
        )
    )
    .drop("fill_name")
)

### 1.4 Unit Cost Null Handling

Impute null `unit_cost` using average margin percentage per product line: `unit_cost = unit_price * (1 - avg_margin/100)`.

In [0]:
margin_lookup = (
    products_df
    .filter(col("unit_cost").isNotNull())
    .withColumn(
        "margin",
        (
            (col("unit_price")-col("unit_cost"))
            /
            col("unit_price")
        )*100
    )
    .groupBy("product_line")
    .agg(
        avg("margin").alias("avg_margin")
    )
)

products_df = (
    products_df
    .join(margin_lookup,"product_line","left")
)

products_df = (
    products_df
    .withColumn(
        "unit_cost",
        when(
            col("unit_cost").isNull(),
            round(
                col("unit_price")
                *
                (1-col("avg_margin")/100),
                2
            )
        )
        .otherwise(col("unit_cost"))
    )
)

### 1.5 Invalid Value Correction

Fix records where `unit_cost > unit_price` (impossible margin) by recalculating cost using the product line's average margin.

In [0]:
products_df = (
    products_df
    .withColumn(
        "unit_cost",
        when(
            col("unit_cost") > col("unit_price"),
            round(
                col("unit_price")
                *
                (1-col("avg_margin")/100),
                2
            )
        )
        .otherwise(col("unit_cost"))
    )
)

### 1.6 Derived Columns

Calculate `margin_pct` as `((unit_price - unit_cost) / unit_price) * 100`.

In [0]:
products_df = (
    products_df
    .withColumn(
        "margin_pct",
        round(
            (
                (col("unit_price")-col("unit_cost"))
                /
                col("unit_price")
            )*100,
            2
        )
    )
)

### 1.7 Remove Duplicates

Drop the original `Product` column, remove `avg_margin`, and deduplicate on `product_number`.

In [0]:
products_df = (
    products_df
    .drop("Product")
    .drop("avg_margin")
    .dropDuplicates(["product_number"])
)

## 2. Retailers Silver Transformation

**Source:** `bronze_retailers` (578 records)  
**Target:** `silver_retailers`

Standardize names, impute nulls using most common values, and deduplicate on `Retailer_code`.

In [0]:
bronze_retailers = spark.table(
    f"{schema}.bronze_retailers"
)

retailers_df = (
    bronze_retailers

    .withColumn(
        "retailer_name",
        initcap(trim(col("Retailer_name")))
    )

    .withColumn(
        "retailer_type",
        initcap(trim(col("Type")))
    )

    .withColumn(
        "country",
        initcap(trim(col("Country")))
    )
)

### 2.1 Retailer Name Imputation

Fill null `retailer_name` with a generated placeholder: `Retailer_<code>`.

In [0]:
retailers_df = (
    retailers_df
    .withColumn(
        "retailer_name",
        when(
            col("retailer_name").isNull(),
            concat(
                lit("Retailer_"),
                col("Retailer_code")
            )
        )
        .otherwise(col("retailer_name"))
    )
)

### 2.2 Country & Type Imputation

Fill null `country` and `retailer_type` with the most common value from the dataset.

In [0]:
most_common_country = (
    retailers_df
    .filter(col("country").isNotNull())
    .groupBy("country")
    .count()
    .orderBy(desc("count"))
    .first()[0]
)

most_common_type = (
    retailers_df
    .filter(col("retailer_type").isNotNull())
    .groupBy("retailer_type")
    .count()
    .orderBy(desc("count"))
    .first()[0]
)

retailers_df = (
    retailers_df

    .withColumn(
        "country",
        coalesce(
            col("country"),
            lit(most_common_country)
        )
    )

    .withColumn(
        "retailer_type",
        coalesce(
            col("retailer_type"),
            lit(most_common_type)
        )
    )
)

### 2.3 Remove Duplicates

Drop original `Type` column and deduplicate on `Retailer_code`.

In [0]:
retailers_df = (
    retailers_df
    .drop("Type")
    .dropDuplicates(["Retailer_code"])
)

## 3. Order Methods Silver Transformation

**Source:** `bronze_order_methods` (12 records)  
**Target:** `silver_order_methods`

Standardize method type names, handle nulls, and deduplicate.

In [0]:
bronze_methods = spark.table(
    f"{schema}.bronze_order_methods"
)

methods_df = (
    bronze_methods
    .withColumn(
        "order_method_type",
        initcap(
            trim(
                col("Order_method_type")
            )
        )
    )
)

### 3.1 Null Handling

Fill null `order_method_type` with the most common method type from the dataset.

In [0]:
most_common_method = (
    methods_df
    .filter(
        col("order_method_type")
        .isNotNull()
    )
    .groupBy("order_method_type")
    .count()
    .orderBy(desc("count"))
    .first()[0]
)

methods_df = (
    methods_df
    .withColumn(
        "order_method_type",
        coalesce(
            col("order_method_type"),
            lit(most_common_method)
        )
    )
)

### 3.2 Remove Duplicates

Deduplicate on `Order_method_code`.

In [0]:
methods_df = (
    methods_df
    .dropDuplicates(
        ["Order_method_code"]
    )
)

## 4. Daily Sales Silver Transformation

**Source:** `bronze_daily_sales` (1,030 records)  
**Target:** `silver_daily_sales` (partitioned by year, month)

Fix sale prices, calculate revenue/discount metrics, and add date dimension columns.

In [0]:
bronze_sales = spark.table(
    f"{schema}.bronze_daily_sales"
)

sales_df = (
    bronze_sales

    .withColumn(
        "unit_sale_price",
        when(
            col("Unit_sale_price").isNull()
            |
            (col("Unit_sale_price")<=0),
            col("Unit_price")
        )
        .otherwise(
            col("Unit_sale_price")
        )
    )
)

### 4.1 Business Calculations

Derive `revenue`, `discount_amount`, and `discount_pct` from quantity and price columns.

In [0]:
sales_df = (
    sales_df

    .withColumn(
        "revenue",
        round(
            col("Quantity")
            *
            col("unit_sale_price"),
            2
        )
    )

    .withColumn(
        "discount_amount",
        round(
            (
                col("Unit_price")
                -
                col("unit_sale_price")
            )
            *
            col("Quantity"),
            2
        )
    )

    .withColumn(
        "discount_pct",
        round(
            (
                (
                    col("Unit_price")
                    -
                    col("unit_sale_price")
                )
                /
                col("Unit_price")
            )*100,
            2
        )
    )
)

### 4.2 Date Dimension Columns

Extract `year`, `month`, `quarter`, and `week` from the `Date` column for partitioning and analytics.

In [0]:
sales_df = (
    sales_df

    .withColumn("year",year("Date"))
    .withColumn("month",month("Date"))
    .withColumn("quarter",quarter("Date"))
    .withColumn("week",weekofyear("Date"))
)

## 5. SCD Type 2 - Product Dimension with History

Implements Slowly Changing Dimension Type 2 to track historical changes in product attributes.

**Features:**
* `effective_from` — Start date of this version
* `effective_to` — End date (NULL for current version)
* `is_current` — Flag indicating current/active version
* `version_number` — Incremental version counter

In [0]:
from pyspark.sql.window import Window

# Add SCD Type 2 metadata columns
products_scd = (
    products_df
    .withColumn("effective_from", current_date())
    .withColumn("effective_to", lit(None).cast("date"))
    .withColumn("is_current", lit(True))
    .withColumn("version_number", lit(1))
)

products_scd.createOrReplaceTempView("products_incoming")

In [0]:
# Initialize silver_products with SCD Type 2 structure
# Writing to ADLS container: /mnt/gosales/silver/
silver_path = "/mnt/gosales/silver"

print("Initializing silver_products with SCD Type 2 structure...")

products_scd.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .option("path", f"{silver_path}/silver_products") \
    .saveAsTable(f"{schema}.silver_products")

print(f"✅ silver_products initialized with {products_scd.count()} records")
print(f"   Location: {silver_path}/silver_products")
print("   Columns: effective_from, effective_to, is_current, version_number")

Initializing silver_products with SCD Type 2 structure...
✅ silver_products initialized with 270 records
   Location: /mnt/gosales/silver/silver_products
   Columns: effective_from, effective_to, is_current, version_number


## SCD Type 2 MERGE Logic

Two-step process:
1. **Expire changed records**: Set `effective_to` and `is_current=False` for records that changed
2. **Insert new versions**: Add new records and updated versions with current metadata

In [0]:
%sql
-- Step 1: Expire old records where data changed
MERGE INTO default.silver_products AS target
USING products_incoming AS source
ON target.product_number = source.product_number
   AND target.is_current = TRUE

WHEN MATCHED AND (
    -- Detect changes in any business attribute
    target.product_name != source.product_name OR
    target.product_brand != source.product_brand OR
    target.product_type != source.product_type OR
    target.Unit_cost != source.Unit_cost OR
    target.Unit_price != source.Unit_price
)
THEN UPDATE SET
    target.effective_to = CURRENT_DATE(),
    target.is_current = FALSE

num_affected_rows,num_updated_rows,num_deleted_rows,num_inserted_rows
0,0,0,0


In [0]:
# Step 2: Insert new and changed records
# Get current max version for each product to increment
from pyspark.sql.window import Window

# Get existing products with their max version
existing_versions = (
    spark.table(f"{schema}.silver_products")
    .groupBy("product_number")
    .agg(max("version_number").alias("max_version"))
)

# Join incoming with existing to determine version numbers
products_to_insert = (
    products_scd
    .join(
        existing_versions,
        "product_number",
        "left"
    )
    .withColumn(
        "version_number",
        when(
            col("max_version").isNull(),
            lit(1)  # New product
        ).otherwise(
            col("max_version") + 1  # Increment version
        )
    )
    .drop("max_version")
)

# Filter to only records that need inserting:
# 1. Brand new products (not in target)
# 2. Changed products (that were just expired)
expired_products = (
    spark.table(f"{schema}.silver_products")
    .filter(
        (col("effective_to") == current_date()) &
        (col("is_current") == False)
    )
    .select("product_number")
    .distinct()
)

# New products = incoming products not in target at all
existing_products = (
    spark.table(f"{schema}.silver_products")
    .select("product_number")
    .distinct()
)

new_products = (
    products_to_insert
    .join(
        existing_products,
        "product_number",
        "left_anti"  # Products in incoming but not in target
    )
)

# Changed products = products that were just expired
changed_products = (
    products_to_insert
    .join(
        expired_products,
        "product_number",
        "inner"  # Products that were expired in step 1
    )
)

# Union new and changed, then append
final_inserts = new_products.union(changed_products)

if final_inserts.count() > 0:
    final_inserts.write \
        .format("delta") \
        .mode("append") \
        .saveAsTable(f"{schema}.silver_products")
    
    print(f"✅ Inserted {final_inserts.count()} new/changed product versions")
else:
    print("ℹ️ No changes detected - no inserts needed")

ℹ️ No changes detected - no inserts needed


## 6. Write Silver Tables

Persist all cleaned DataFrames to Delta tables with schema evolution enabled.

* `silver_products` — Uses SCD Type 2 (handled via MERGE above)
* `silver_retailers` — Overwrite with mergeSchema
* `silver_order_methods` — Overwrite with mergeSchema
* `silver_daily_sales` — Partitioned by `year`, `month`

In [0]:
# Note: Products table uses SCD Type 2 (handled above via MERGE + INSERT)
# All silver tables written to ADLS: /mnt/gosales/silver/
if not spark.catalog.tableExists(f"{schema}.silver_products"):
    print("Creating silver_products table with SCD Type 2 structure...")
    products_scd.write \
        .format("delta") \
        .mode("overwrite") \
        .option("overwriteSchema","true") \
        .option("mergeSchema","true") \
        .option("path", f"{silver_path}/silver_products") \
        .saveAsTable(f"{schema}.silver_products")
    print("✅ silver_products table created")
else:
    print("ℹ️ silver_products already exists - using SCD Type 2 MERGE logic above")

retailers_df.write \
.format("delta") \
.mode("overwrite") \
.option("overwriteSchema","true") \
.option("mergeSchema","true") \
.option("path", f"{silver_path}/silver_retailers") \
.saveAsTable(f"{schema}.silver_retailers")

methods_df.write \
.format("delta") \
.mode("overwrite") \
.option("overwriteSchema","true") \
.option("mergeSchema","true") \
.option("path", f"{silver_path}/silver_order_methods") \
.saveAsTable(f"{schema}.silver_order_methods")

sales_df.write \
.format("delta") \
.partitionBy("year","month") \
.mode("overwrite") \
.option("overwriteSchema","true") \
.option("mergeSchema","true") \
.option("path", f"{silver_path}/silver_daily_sales") \
.saveAsTable(f"{schema}.silver_daily_sales")

print(f"✅ All silver tables written to: {silver_path}/")

ℹ️ silver_products already exists - using SCD Type 2 MERGE logic above
✅ All silver tables written to: /mnt/gosales/silver/


## 7. Delta Optimization

Compact small files using `OPTIMIZE` for improved query performance.

In [0]:
%sql
OPTIMIZE default.silver_products;
OPTIMIZE default.silver_retailers;
OPTIMIZE default.silver_order_methods;
OPTIMIZE default.silver_daily_sales;

path,metrics
dbfs:/mnt/gosales/silver/silver_daily_sales,"List(0, 0, List(null, null, 0.0, 0, 0), List(null, null, 0.0, 0, 0), 12, null, 0, 12, 12, true, 0, 0, 1782210606835, 1782210607337, 4, 0, null, List(0, 0), 19, 19, 0, 0, null)"


### 7.1 Z-Ordering

Apply Z-ORDER on frequently filtered columns for co-locality:
* `silver_products` → `product_number`
* `silver_retailers` → `Retailer_code`
* `silver_daily_sales` → `Retailer_code`, `Product_number`

In [0]:
%sql
OPTIMIZE default.silver_products
ZORDER BY (product_number);

OPTIMIZE default.silver_retailers
ZORDER BY (Retailer_code);

OPTIMIZE default.silver_daily_sales
ZORDER BY
(
Retailer_code,
Product_number
);

path,metrics
dbfs:/mnt/gosales/silver/silver_daily_sales,"List(0, 0, List(null, null, 0.0, 0, 0), List(null, null, 0.0, 0, 0), 12, List(minCubeSize(107374182400), List(0, 0), List(12, 90023), 0, List(0, 0), 0, null), 0, 12, 12, false, 0, 0, 1782210612582, 1782210613538, 4, 0, null, List(0, 0), 19, 19, 0, 0, null)"


## 8. Data Quality Checks

Validate silver tables for null thresholds, SCD Type 2 integrity, and record counts.

## Validate SCD Type 2 History

Query to show historical versions of products with change tracking.

In [0]:
%sql
-- Show products with their historical versions
SELECT 
    product_number,
    product_name,
    product_brand,
    product_type,
    Unit_cost,
    Unit_price,
    version_number,
    effective_from,
    effective_to,
    is_current,
    CASE 
        WHEN is_current THEN '✅ Current'
        ELSE '📋 Historical'
    END AS status
FROM default.silver_products
WHERE product_number IN (
    -- Find products with multiple versions
    SELECT product_number 
    FROM default.silver_products 
    GROUP BY product_number 
    HAVING COUNT(*) > 1
)
ORDER BY product_number, version_number DESC
LIMIT 50

product_number,product_name,product_brand,product_type,Unit_cost,Unit_price,version_number,effective_from,effective_to,is_current,status


In [0]:
%sql
-- SCD Type 2 Summary Statistics
SELECT
    COUNT(DISTINCT product_number) AS total_unique_products,
    COUNT(*) AS total_records_with_history,
    SUM(CASE WHEN is_current THEN 1 ELSE 0 END) AS current_versions,
    SUM(CASE WHEN NOT is_current THEN 1 ELSE 0 END) AS historical_versions,
    MAX(version_number) AS max_version_number,
    COUNT(DISTINCT CASE WHEN version_number > 1 THEN product_number END) AS products_with_changes
FROM default.silver_products

total_unique_products,total_records_with_history,current_versions,historical_versions,max_version_number,products_with_changes
270,270,270,0,1,0


## 🧪 SCD Type 2 Demo - Simulating Product Changes

This demo shows how SCD Type 2 tracks historical changes when product attributes are updated.

In [0]:
# Demo: Simulate a product price change to show SCD Type 2 in action
from pyspark.sql.functions import *

print("🚧 DEMO: Simulating price changes for 3 products...\n")

# Get 3 sample products to modify
sample_products = (
    spark.table(f"{schema}.silver_products")
    .filter(col("is_current") == True)
    .limit(3)
)

# Show original values
print("📊 Original Product Data:")
sample_products.select(
    "product_number", "product_name", "Unit_price", "version_number", "is_current"
).show(truncate=False)

# Simulate price increases
updated_products = (
    sample_products
    .withColumn("Unit_price", round(col("Unit_price") * 1.15, 2))  # 15% price increase
    .withColumn("effective_from", current_date())
    .withColumn("effective_to", lit(None).cast("date"))
    .withColumn("is_current", lit(True))
    .withColumn("version_number", lit(1))  # Will be incremented by SCD logic
)

print("\n⚡ Updated Product Data (15% price increase):")
updated_products.select(
    "product_number", "product_name", "Unit_price", "version_number", "is_current"
).show(truncate=False)

# Create temp view for MERGE
updated_products.createOrReplaceTempView("products_incoming")

print("\n✅ Ready to run SCD Type 2 MERGE (cells 42-43)")

🚧 DEMO: Simulating price changes for 3 products...

📊 Original Product Data:
+--------------+-----------------+----------+--------------+----------+
|product_number|product_name     |Unit_price|version_number|is_current|
+--------------+-----------------+----------+--------------+----------+
|135110        |Ranger Vision    |163.48    |1             |true      |
|2110          |Trailchef Canteen|12.92     |1             |true      |
|125110        |Venue            |71.93     |1             |true      |
+--------------+-----------------+----------+--------------+----------+


⚡ Updated Product Data (15% price increase):
+--------------+-----------------+----------+--------------+----------+
|product_number|product_name     |Unit_price|version_number|is_current|
+--------------+-----------------+----------+--------------+----------+
|135110        |Ranger Vision    |188.0     |1             |true      |
|2110          |Trailchef Canteen|14.86     |1             |true      |
|125110     

## 10. Final Silver Validation

Run comprehensive null validation with percentage thresholds (reject if >5% nulls in critical columns) across all silver tables.

In [0]:
# Null validation with percentage thresholds
NULL_THRESHOLD_PCT = 5.0  # Reject if > 5% nulls in critical columns

tables_to_validate = [
    ("silver_products", ["product_number", "product_name"]),
    ("silver_retailers", ["Retailer_code", "retailer_name"]),
    ("silver_order_methods", ["Order_method_code"]),
    ("silver_daily_sales", ["Retailer_code", "Product_number", "Date"])
]

print("\n===== SILVER LAYER DATA QUALITY VALIDATION =====")

for table_name, critical_columns in tables_to_validate:
    print(f"\n📊 Validating {table_name}...")
    
    # Get schema once (fixes Spark Connect performance issue)
    table_df = spark.table(f"{schema}.{table_name}")
    total_count = table_df.count()
    all_columns = table_df.columns
    
    # Build null count query
    null_checks = ",".join([
        f"SUM(CASE WHEN {c} IS NULL THEN 1 ELSE 0 END) AS {c}"
        for c in all_columns
    ])
    
    null_counts = spark.sql(f"""
        SELECT {null_checks}
        FROM {schema}.{table_name}
    """)
    
    # Check thresholds
    null_dict = null_counts.first().asDict()
    quality_passed = True
    
    for col_name, null_count in null_dict.items():
        null_pct = (null_count / total_count * 100) if total_count > 0 else 0
        
        if null_count > 0:
            status = "⚠️" if col_name in critical_columns else "ℹ️"
            print(f"  {status} {col_name}: {null_count} nulls ({null_pct:.2f}%)")
            
            # Check critical columns against threshold
            if col_name in critical_columns and null_pct > NULL_THRESHOLD_PCT:
                print(f"    ❌ FAILED: Exceeds {NULL_THRESHOLD_PCT}% threshold!")
                quality_passed = False
    
    if quality_passed:
        print(f"  ✅ {table_name} PASSED quality checks")
    else:
        print(f"  ❌ {table_name} FAILED quality checks")
        raise Exception(f"Data quality check failed for {table_name}: Critical columns exceed {NULL_THRESHOLD_PCT}% null threshold")

print("\n✅ All Silver tables passed data quality validation!")


===== SILVER LAYER DATA QUALITY VALIDATION =====

📊 Validating silver_products...
  ℹ️ effective_to: 270 nulls (100.00%)
  ✅ silver_products PASSED quality checks

📊 Validating silver_retailers...
  ✅ silver_retailers PASSED quality checks

📊 Validating silver_order_methods...
  ✅ silver_order_methods PASSED quality checks

📊 Validating silver_daily_sales...
  ✅ silver_daily_sales PASSED quality checks

✅ All Silver tables passed data quality validation!
